In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import pyogrio
import os
import win32com.client as com

c:\Users\Roberto Ponce López\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
FOLDER = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"
FOLDER_HEADWAYS = r"Insumos IMEPLAN\Transporte Publico\FRECUENCIAS"
FOLDER_NETWORK = r"Red Base GDL\RedBase Final\version base para rezonificacion"

### Get Line Routes from Visum (800)

In [4]:
#Red base GDL (con 2,203 zonas)
red_base = os.path.join(FOLDER, FOLDER_NETWORK, "RedFinal - rezonificacion - v2.ver")
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

In [30]:
line_routes = pd.DataFrame({
    "ID_Visum": [i[1] for i in Visum.Net.LineRoutes.GetMultiAttValues("ID")],
    "Name": [i[1] for i in Visum.Net.LineRoutes.GetMultiAttValues("Name")],
    "TSysCode": [i[1] for i in Visum.Net.LineRoutes.GetMultiAttValues("TSysCode")],
})
print(f"Total line routes: {len(line_routes)}")

Total line routes: 800


### Leer frecuencias
- hora max demanda y hora valle

In [31]:
headways = pd.read_excel(os.path.join(FOLDER, FOLDER_HEADWAYS, "Cruce_rutas_Visum_frecuencias_PARA_ENTREGAR.xlsx"))
headways = headways.rename(columns={"Nombre_LineRoute_Visum": "Name"})
headways = headways.drop_duplicates(subset="Name", keep="first")


line_routes = line_routes.merge(
    headways[["Name", "FRECUENCIA_HMD_FINAL", "FRECUENCIA_HORA_VALLE_FINAL"]], 
    on=["Name"], 
    how="left"
)
print(f"Total line routes with headways: {line_routes['FRECUENCIA_HMD_FINAL'].notnull().sum()}")
print(f"Total line routes with headways: {line_routes['FRECUENCIA_HORA_VALLE_FINAL'].notnull().sum()}")

Total line routes with headways: 800
Total line routes with headways: 800
